In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
%load_ext autoreload
%autoreload 2

import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import ActiveSparringReset
from src.rl.reward_shapers import Stage2Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")


def make_sparring_env(
    red_players: int = 1,
    blue_players: int = 1,
    opponent_accel: float = 1800.0,
    kickoff_prob: float = 0.15,
    max_steps: int = 600,  # 10s isolated possession rounds
):
    def _init():
        red_coord = TeamHeuristicCoordinator(team="red")
        blue_coord = TeamHeuristicCoordinator(team="blue")
        roster = []

        # 1. Primary RL Agent
        roster.append(PlayerSlot(
            team="red",
            stats=PlayerStats(name="RL_Agent", accel=3200.0)
        ))

        # 2. Additional Teammates (for N vs M scaling)
        for i in range(1, red_players):
            roster.append(PlayerSlot(
                team="red",
                stats=PlayerStats(name=f"Red_Bot_{i}", accel=3000.0),
                controller=HeuristicBotController(red_coord)
            ))

        # 3. Sparring Opponents
        for j in range(blue_players):
            roster.append(PlayerSlot(
                team="blue",
                stats=PlayerStats(name=f"Blue_Bot_{j + 1}", accel=opponent_accel),
                controller=HeuristicBotController(blue_coord)
            ))

        match_cfg = MatchConfig(
            mode=ClassicMatchMode(
                time_limit=10.0,
                score_limit=1,  # 1 goal finishes the possession round
                kickoff_timeout=5.0,
            ),
            roster=roster,
            time_limit=10.0,
            score_limit=1,
        )

        return HaxballGymEnv(
            match_config=match_cfg,
            reward_shaper=Stage2Reward(team="red"),
            reset_strategy=ActiveSparringReset(kickoff_prob=kickoff_prob),
            max_steps=max_steps,
        )
    return _init


NUM_ENVS = 16
OBS_DIM = 80

# Training Environments: 10s isolated sparring possessions
train_envs = gym.vector.AsyncVectorEnv(
    [make_sparring_env(
        red_players=1,
        blue_players=1,
        opponent_accel=3000.0,
        kickoff_prob=0.15,
        max_steps=600,
    ) for _ in range(NUM_ENVS)]
)

# Evaluation Environment: 10s fixed assessment
eval_env = make_sparring_env(
    red_players=1,
    blue_players=1,
    opponent_accel=3000.0,
    kickoff_prob=0.15,
    max_steps=600,
)()

# Load Stage 1 Pre-Trained Weights
model = ActorCritic(obs_dim=OBS_DIM).to(device)
stage1_ckpt = "models/stage1/best_model.pt"
model.load_state_dict(torch.load(stage1_ckpt, map_location=device, weights_only=False))
print(f"✅ Loaded base motor policy from {stage1_ckpt}")

# Execute PPO Training
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2/phaseA",
    lr_initial=1e-4,          # Fine-tuning rate
    lr_final=5e-6,
    ent_coef_initial=0.003,
    ent_coef_final=0.0005,
)

train_envs.close()
eval_env.close()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
⚡ Device: cuda


✅ Loaded base motor policy from models/stage1/best_model.pt
🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:   0.0% (0/50) | Conceded:  26.0% (13/50) | Net: -13 | Touch:  64.0% | Avg Steps: 509.1 | Mean Reward: -21.17
   ⭐ New verified best model saved: models/stage2/phaseA/best_model.pt
      [Net: -13 | Scored: 0.0% | Reward: -21.17 | Speed: 509.1 steps]


📊 [EVALUATION @ Step  200704] Scored:   6.0% (3/50) | Conceded:  28.0% (14/50) | Net: -11 | Touch:  84.0% | Avg Steps: 508.2 | Mean Reward: -18.69
   ⭐ New verified best model saved: models/stage2/phaseA/best_model.pt
      [Net: -11 | Scored: 6.0% | Reward: -18.69 | Speed: 508.2 steps]


📊 [EVALUATION @ Step  303104] Scored:   4.0% (2/50) | Conceded:  26.0% (13/50) | Net: -11 | Touch:  66.0% | Avg Steps: 502.1 | Mean Reward: -15.20

📊 [EVALUATION @ Step  401408] Scored:   8.0% (4/50) | Conceded:  12.0% (6/50) | Net:  -2 | Touch:  84.0% | Avg Steps: 545.9 | Mean Reward:   1.07
   ⭐ New v

In [ ]:
# Training Environments: 10s isolated sparring possessions
train_envs = gym.vector.AsyncVectorEnv(
    [make_sparring_env(
        red_players=1,
        blue_players=1,
        opponent_accel=3200.0,
        kickoff_prob=0.15,
        max_steps=600,
    ) for _ in range(NUM_ENVS)]
)

# Evaluation Environment: 10s fixed assessment
eval_env = make_sparring_env(
    red_players=1,
    blue_players=1,
    opponent_accel=3200.0,
    kickoff_prob=0.15,
    max_steps=600,
)()

# Load Stage 1 Pre-Trained Weights
model = ActorCritic(obs_dim=OBS_DIM).to(device)
stage2_ckpt = "models/stage2/phaseA/best_model.pt"
model.load_state_dict(torch.load(stage2_ckpt, map_location=device, weights_only=False))
print(f"✅ Loaded base motor policy from {stage2_ckpt}")

# Execute PPO Training
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2/phaseB",
    lr_initial=1e-4,          # Fine-tuning rate
    lr_final=5e-6,
    ent_coef_initial=0.003,
    ent_coef_final=0.0005,
)

train_envs.close()
eval_env.close()

✅ Loaded base motor policy from models/stage2/phaseA/best_model.pt
🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:  12.0% (6/50) | Conceded:  16.0% (8/50) | Net:  -2 | Touch:  86.0% | Avg Steps: 513.8 | Mean Reward:  -4.19
   ⭐ New verified best model saved: models/stage2/phaseB/best_model.pt
      [Net: -2 | Scored: 12.0% | Reward: -4.19 | Speed: 513.8 steps]


📊 [EVALUATION @ Step  200704] Scored:  16.0% (8/50) | Conceded:  10.0% (5/50) | Net:  +3 | Touch:  94.0% | Avg Steps: 525.6 | Mean Reward:  10.21
   ⭐ New verified best model saved: models/stage2/phaseB/best_model.pt
      [Net: +3 | Scored: 16.0% | Reward: 10.21 | Speed: 525.6 steps]


📊 [EVALUATION @ Step  303104] Scored:  18.0% (9/50) | Conceded:  26.0% (13/50) | Net:  -4 | Touch:  88.0% | Avg Steps: 453.9 | Mean Reward:   7.68

📊 [EVALUATION @ Step  401408] Scored:   8.0% (4/50) | Conceded:  30.0% (15/50) | Net: -11 | Touch:  78.0% | Avg Steps: 486.1 | Mean Reward: -14.73

📊 [EV

# Testing

In [6]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load RL models
obs_dim = 80
stage2_model = ActorCritic(obs_dim).to(device)
stage2_model.load_state_dict(torch.load("models/stage1/best_model.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage2_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage2_model, team="blue")



In [7]:
# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 3 goals
   Episode 2: 0 goals
   Episode 3: 0 goals
   Episode 4: 6 goals
   Episode 5: 8 goals
📊 Average Scoring Rate: 3.40 goals / 60.0s



3.4

In [8]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (1 Matches)
✅ Completed in 10.57s
🏆 Series Outcome (Wins): RED 0 | BLUE 0 | DRAWS 1
⚽ Avg Goals / Match:     RED 2.00 | BLUE 2.00



In [9]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: RED WINS! 🎉 (3 - 1)
Replay saved to: renders/arena/2026-08-22_12-24-27_match_1.html



In [10]:
from src.rl.benchmarker import render_solo_drill

device = torch.device("cpu")
model = ActorCritic(obs_dim=80).to(device)
model.load_state_dict(torch.load("models/stage2/best_model.pt", map_location=device, weights_only=False))

agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 2 Goals Scored
   Episode 2 Finished: 2 Goals Scored
   Episode 3 Finished: 3 Goals Scored
🏆 Overall: 2.33 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-22_12-27-00_solo_drill.html

